# X Community Notes — full public data dump downloader

Downloads the current daily snapshot of the [Community Notes public data](https://communitynotes.x.com/guide/en/under-the-hood/download-data) straight from X's CDN (no login, no API key).

**What you get** (zips, each containing one TSV with a header row):

| dataset | contents | size (2026-08-25, zipped) |
|---|---|---|
| `notes` | every surviving note ever written (full text, classification, tweetId) | ~0.5 GB, 3 parts |
| `noteRatings` | every rating of every note | several GB, multiple parts |
| `noteStatusHistory` | status timeline of every *scored* note (survives note deletion) | ~0.18 GB |
| `userEnrollment` | anonymized contributor enrollment states | ~0.07 GB |

**Things to know before you build on this:**
- Snapshots are **cumulative** — today's dump contains all data since the January 2021 launch (with a 48 h lag), so you don't need old dumps to get history…
- …**except**: deleted notes disappear from all future dumps (only `noteStatusHistory` keeps their metadata), and note statuses change over time. For longitudinal work you need archived snapshots.
- X keeps only a **rolling ~7-day window** of past snapshots on the CDN. Older dates → 404. If you need snapshots, archive them yourself (daily cron on this script) or use third-party archives (see the last section).
- **License**: the data has no explicit open license. X states all contributions are "publicly available so that anyone has free access to analyze the data"; use is governed by the [X Terms of Service](https://x.com/en/tos). Participant IDs are anonymized/hashed. The [scoring source code](https://github.com/twitter/communitynotes) is Apache-2.0 (that license covers the *code*, not the data).

In [1]:
import pandas as pd

from cn_downloader import (
    DATASETS,
    DEFAULT_OUT_DIR,
    download_snapshot,
    find_latest_snapshot,
    list_available_snapshots,
)

## 1. Which snapshots are currently on the CDN?

Probes `https://ton.twimg.com/birdwatch-public-data/YYYY/MM/DD/notes/notes-00000.zip` for the last 10 days (~20 s: the CDN dislikes rapid-fire requests, so there is a polite delay between probes).

In [2]:
available = list_available_snapshots()
print("Snapshots currently downloadable:")
for d in available:
    print(f"  {d:%Y-%m-%d}")

Snapshots currently downloadable:
  2026-08-19
  2026-08-20
  2026-08-21
  2026-08-22
  2026-08-23
  2026-08-24
  2026-08-25


## 2. Download the current full dump

Downloads all four datasets of the latest snapshot into `data/raw/YYYY-MM-DD/`, verifies zip integrity, and writes a `manifest.json`. Re-running skips files that are already complete. Expect **several GB** (the ratings dominate) — on a fast connection this takes on the order of tens of minutes the first time.

To grab only the small datasets first, use e.g. `datasets=["notes", "noteStatusHistory", "userEnrollment"]`.

In [3]:
manifest = download_snapshot()  # date=None -> latest; datasets=None -> all four
snapshot_date = manifest["snapshot_date"]

[notes]
  notes part 0: 3.2 MB


  notes part 1: 3.2 MB


  notes part 2: 481.1 MB


[noteRatings]
  noteRatings part 0: 1,430.9 MB


  noteRatings part 1: 1,452.7 MB


...

noteStatusHistory-00000.zip:   0%|          | 0.00/179M [00:00<?, ?B/s]

[userEnrollment]


  userEnrollment part 0: 70.9 MB


userEnrollment-00000.zip:   0%|          | 0.00/70.9M [00:00<?, ?B/s]

Done: 12.27 GB (zipped) in /Users/bertik/PhD/community-notes/data/raw/2026-08-25


## 3. Sanity check — peek at each table

`pandas` reads the single-member zips directly; no need to extract. IDs are read as strings so 64-bit snowflake IDs survive intact.

In [4]:
for dataset, files in manifest["files"].items():
    first_part = files[0]["path"]
    df = pd.read_csv(first_part, sep="\t", nrows=3, dtype=str)
    print(f"=== {dataset} ({len(files)} part(s)) — {df.shape[1]} columns ===")
    print(", ".join(df.columns[:12]) + (" …" if df.shape[1] > 12 else ""))
    display(df.head(3))

=== notes (3 part(s)) — 24 columns ===
noteId, noteAuthorParticipantId, createdAtMillis, tweetId, classification, believable, harmful, validationDifficulty, misleadingOther, misleadingFactualError, misleadingManipulatedMedia, misleadingOutdatedInformation …


,noteId,noteAuthorParticipantId,createdAtMillis,tweetId,classification,believable,harmful,validationDifficulty,misleadingOther,misleadingFactualError,...,misleadingSatire,notMisleadingOther,notMisleadingFactuallyCorrect,notMisleadingOutdatedButNotWhenWritten,notMisleadingClearlySatire,notMisleadingPersonalOpinion,trustworthySources,summary,isMediaNote,isCollaborativeNote
0,2089379163832508416,F75B04680623CD9A452496D4AB69A20ED730CE93652C71...,1786981943226,254231749758943233,MISINFORMED_OR_POTENTIALLY_MISLEADING,NaN,NaN,NaN,0,0,...,0,0,0,0,0,0,0,This 2012 tweet was not referring to OpenAI’s ...,0,1
1,2070196127807541737,F75B04680623CD9A452496D4AB69A20ED730CE93652C71...,1782408276566,1146442848914608128,NOT_MISLEADING,NaN,NaN,NaN,0,0,...,0,0,0,0,0,0,0,This 2019 post notes something missing from Me...,0,1
2,2033333592244596856,F75B04680623CD9A452496D4AB69A20ED730CE93652C71...,1773619630910,1261399556702351360,NOT_MISLEADING,NaN,NaN,NaN,0,0,...,0,0,0,0,0,0,0,This post solicits testimonials from past clie...,0,1


=== noteRatings (9 part(s)) — 35 columns ===
noteId, raterParticipantId, createdAtMillis, version, agree, disagree, helpful, notHelpful, helpfulnessLevel, helpfulOther, helpfulInformative, helpfulClear …


,noteId,raterParticipantId,createdAtMillis,version,agree,disagree,helpful,notHelpful,helpfulnessLevel,helpfulOther,...,notHelpfulArgumentativeOrBiased,notHelpfulOffTopic,notHelpfulSpamHarassmentOrAbuse,notHelpfulIrrelevantSources,notHelpfulOpinionSpeculation,notHelpfulNoteNotNeeded,ratedOnTweetId,ratingSourceBucketed,suggestion,suggestionId
0,1352796878438424576,A8CE8EFCAFC6F6228EEFC20E211071E6DCB40B97A050B6...,1611623203338,1,0,0,0,1,NaN,0,...,0,0,0,0,0,0,-1,DEFAULT,NaN,-1
1,1352796878438424576,457AA0AE4D242090FF340BB7E0154AC3DF30904BEEF5C5...,1611852744990,1,1,0,1,0,NaN,0,...,0,0,0,0,0,0,-1,DEFAULT,NaN,-1
2,1352796878438424576,00647715E8F0DD5D4EFFE5210FCD0DC741C3B921218124...,1611388222120,1,1,0,1,0,NaN,0,...,0,0,0,0,0,0,-1,DEFAULT,NaN,-1


=== noteStatusHistory (1 part(s)) — 23 columns ===
noteId, noteAuthorParticipantId, createdAtMillis, timestampMillisOfFirstNonNMRStatus, firstNonNMRStatus, timestampMillisOfCurrentStatus, currentStatus, timestampMillisOfLatestNonNMRStatus, mostRecentNonNMRStatus, timestampMillisOfStatusLock, lockedStatus, timestampMillisOfRetroLock …


,noteId,noteAuthorParticipantId,createdAtMillis,timestampMillisOfFirstNonNMRStatus,firstNonNMRStatus,timestampMillisOfCurrentStatus,currentStatus,timestampMillisOfLatestNonNMRStatus,mostRecentNonNMRStatus,timestampMillisOfStatusLock,...,currentExpansionStatus,currentGroupStatus,currentDecidedBy,currentModelingGroup,timestampMillisOfMostRecentStatusChange,timestampMillisOfNmrDueToMinStableCrhTime,currentMultiGroupStatus,currentModelingMultiGroup,timestampMinuteOfFinalScoringOutput,timestampMillisOfFirstNmrDueToMinStableCrhTime
0,1352796878438424576,B021517F4F8F9F372DCD70BB8F8A21A80C12D7EFDE8B2F...,1611366884227,NaN,NaN,1721920068424,NEEDS_MORE_RATINGS,NaN,NaN,1674003109873,...,NEEDS_MORE_RATINGS,NEEDS_MORE_RATINGS,CoreModel (v1.1),13,-1,NaN,NaN,NaN,29791018,NaN
1,1353415873227177985,D4C9EDB464DF2D4A6D8F68CA8D1F86226B50EB9D3B0F33...,1611514464087,NaN,NaN,1721920068424,NEEDS_MORE_RATINGS,NaN,NaN,1674003109873,...,NEEDS_MORE_RATINGS,NEEDS_MORE_RATINGS,CoreModel (v1.1),13,-1,NaN,NaN,NaN,29791018,NaN
2,1354586938863443971,C0AF45F4C4B2240E7AB31456957E5D770FF7AFA13627C7...,1611793667892,NaN,NaN,1781032339438,NEEDS_MORE_RATINGS,NaN,NaN,1674003109873,...,NEEDS_MORE_RATINGS,NaN,CoreModel (v1.1),NaN,-1,-1,NaN,NaN,29791018,NaN


=== userEnrollment (1 part(s)) — 8 columns ===
participantId, enrollmentState, successfulRatingNeededToEarnIn, timestampOfLastStateChange, timestampOfLastEarnOut, modelingPopulation, modelingGroup, numberOfTimesEarnedOut


,participantId,enrollmentState,successfulRatingNeededToEarnIn,timestampOfLastStateChange,timestampOfLastEarnOut,modelingPopulation,modelingGroup,numberOfTimesEarnedOut
0,544F5C7BB68BCC9AC3E28D9E8AEDE7AF4101B5B7A4A2B3...,newUser,5,1730339155731,1,CORE,13.0,0
1,B2D8708DD64F4A263D237C309DDFC40F96962E5B6CF8E2...,newUser,5,1711877500977,1,CORE,13.0,0
2,C784F04F26E124F4D6EC01658D8F5565005D3092741FB3...,earnedIn,12,1773125665610,1737954217871,CORE,13.0,1


In [5]:
# Full row counts (streams through every part; a few minutes for the ratings)
for dataset, files in manifest["files"].items():
    n = 0
    for f in files:
        for chunk in pd.read_csv(f["path"], sep="\t", usecols=[0], dtype=str, chunksize=2_000_000):
            n += len(chunk)
    print(f"{dataset:>20}: {n:>12,} rows")

               notes:    3,003,039 rows


         noteRatings:  229,976,445 rows


   noteStatusHistory:    3,202,860 rows


      userEnrollment:    1,479,675 rows


## 4. Past snapshots

Any date still inside the CDN's ~7-day window can be fetched explicitly:

```python
import datetime as dt
download_snapshot(date=dt.date(2026, 8, 19))  # oldest date still available on 2026-08-25
```

For anything older than the window:
- **Zenodo archive (CC-BY-4.0)** covering **2021-01-23 → 2025-01-23**: monthly rating files, notes with language detection, and interaction graphs, from the paper *"From Birdwatch to Community Notes"* — [zenodo.org/records/16761304](https://zenodo.org/records/16761304)
- Early Birdwatch-era scrapes: [github.com/shiruken/birdwatch-data](https://github.com/shiruken/birdwatch-data)
- Going forward, the durable option is to run this downloader daily (cron/launchd) and keep your own snapshot archive — that is the only way to track note-status evolution and capture notes before deletion.